<h1 class="alert  alert-block alert-info">Azure OpenAI Embedding<h1>

In [ ]:
from langchain_openai import AzureChatOpenAI, ChatOpenAI, AzureOpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

endpoint = os.getenv("AZURE_ENDPOINT")
model = os.getenv("AZURE_OPENAI_EMBEDDING_LARGE_MODEL")

subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

openai_embedding_llm = AzureOpenAIEmbeddings( 
        api_version=api_version,
        azure_deployment=model,
   )

<h2 class="alert alert-block alert-info">Load data from file</h2>

<div class="alert alert-block alert-info">File path: /input_data/sample_stories.txt</div>

In [ ]:
file_path = "../../input_data/sample_stories.txt"
with open(file=file_path, mode="r") as file:
    lines = [ line.strip() for line in file.readlines() if line.strip()]
lines

<h2 class="alert alert-block alert-info">Using Azure OpenAI Embedding</h2>

In [ ]:
embedding = openai_embedding_llm.embed_documents(lines)
print("Number of docs: ",len(embedding))
print("Dims of each doc: ", len(embedding[0]))

<h2 class="alert alert-block alert-info">Using FAISS (Facebook AI Similarity Search)</h2>

<img src="../../Excalidraw/FAISS.png" width=500px>

<div class="alert alert-info">Load if previous embeddings is stored locally. </div>

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

faiss_folder_name = "faiss_index_storage"
if os.path.exists(faiss_folder_name):
    FAISS.load_local(faiss_folder_name, embeddings=openai_embedding_llm, allow_dangerous_deserialization=True)


<div class="alert alert-block alert-info">
1. FAISS used Euclidean (L2) distance.</br>
2. Azure OpenAI used Cosine Similarity (values are [-1, 1]) 

</div>

In [ ]:
vector_db = FAISS.from_texts(lines, openai_embedding_llm, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)
vector_db.save_local("faiss_index_storage")

<div class="alert alert-success">Output of similarity search: </br>
[(Document(id='4f94abca-39b5-4050-ac4d-74f9f4a2faea', metadata={}, page_content='LangChain is a framework for developing LLM applications.'),
  np.float32(0.6720254)),
 (Document(id='9328e848-49b0-4ac3-88d8-838c2048f315', metadata={}, page_content='Python is the primary language used for AI development.'),
  np.float32(0.16530834))]

doc[0].page_content, doc[1]

LangChain is a framework for developing LLM applications. 0.37124735</br>
Python is the primary language used for AI development. 0.8631096
</div> 

In [ ]:
docs = vector_db.similarity_search_with_relevance_scores(query="Explain LangChain", k=2)
for doc in docs:
    print(doc[0].page_content, doc[1])

<div class="alert alert-success">Output of similarity search: </br>
[(Document(id='4f94abca-39b5-4050-ac4d-74f9f4a2faea', metadata={}, page_content='LangChain is a framework for developing LLM applications.'),
  np.float32(0.6720254)),
 (Document(id='9328e848-49b0-4ac3-88d8-838c2048f315', metadata={}, page_content='Python is the primary language used for AI development.'),
  np.float32(0.16530834))]

doc[0].page_content, doc[1]

LangChain is a framework for developing LLM applications. 0.6720254</br>
Python is the primary language used for AI development. 0.16530834
</div> 

In [ ]:
docs = vector_db.similarity_search_with_score(query="What is LangChain", k=2)
docs